# What the Collapse Predictor Actually Buys You: Measured Compute Cost + Screening Efficiency

## Why this exists

`locked-results.md` §3a/§3c/§3g establish that the layer-30 classifier predicts folding collapse
(AUC 0.78 full-sequence, 0.74 from the first 30 residues, 95.8% top-decile precision). What none of
that says is **what it saves you**. That gap is the difference between "we have a working
classifier" and a *decision-aware* result — which is what AI4DD Track 4 ("decision-aware metrics")
and Track 5 ("Multi-Objective and Resource-Constrained Discovery") actually ask for.

A preliminary screening curve already exists (`analysis/screening_efficiency.py`, zero-GPU, N=50
held-out): folding only the top 70% of candidates retains ~95% of the foldable ones. But that number
has two honest holes this notebook closes:

1. **No wall-clock cost was ever measured anywhere in this project.** Without it you can only say
   "30% fewer *folds*", never "X% less *compute*", because nobody knows how expensive a fold is
   relative to a generation. This notebook measures it.
2. **That curve used full-sequence features.** The actual early-abort setting is *truncated*
   features (lower AUC), so 30% is an upper bound. This notebook computes the curve at both.

## What it measures

**Three timed components, with correct CUDA methodology** — `torch.cuda.synchronize()` around every
measurement (CUDA kernels are asynchronous; timing without synchronizing measures queue-submission,
not execution, and is simply wrong) and discarded warm-up iterations (first calls pay one-time
cuDNN autotuning and allocator costs).

- **Generation** — per-sequence and per-token, so the cost of stopping early is derivable.
- **ESMFold** — per-sequence, recorded *with sequence length*, because folding cost grows steeply
  with length and a single average would be misleading.
- **Classifier inference** — one truncated forward pass + scaler + PCA + RandomForest.

**Then an honest screening curve** at full / 30 / 20 / 10 residues, built on **out-of-fold**
predictions via `cross_val_predict` with scaler+PCA+RF inside a `Pipeline`, so no fold's
preprocessing ever sees its own test data. (The N=50 preliminary used a classifier trained on a
separate pool; this is the cleaner within-pool construction.)

**Finally an end-to-end model** combining both: for a triage policy that generates `K` residues,
runs the classifier, and only finishes+folds the survivors, total cost is

    cost(K, keep_rate) = T_gen(K) + T_clf(K) + keep_rate x [ T_gen(full) - T_gen(K) + T_fold ]

compared against the baseline `T_gen(full) + T_fold` for every candidate.

## Honest scope, stated before the numbers

- Times are **hardware-specific** (whatever Kaggle assigns — T4 or P100). The *ratio* between
  components transfers; the absolute seconds do not. Report ratios, and state the GPU.
- ESMFold here folds one sequence at a time, matching how every notebook in this project used it.
  Batched folding would change absolute throughput (not the qualitative conclusion).
- "Foldable" remains ESMFold pLDDT >= 60 — a confidence proxy, unchanged from the rest of the project.
- Savings assume the classifier's cost is paid on every candidate (it is) and that aborted
  candidates are discarded, not regenerated.

Kaggle setup: Accelerator = **GPU T4 x1**, Internet = **ON**. Expect ~45–70 minutes
(N=200 generate + fold + timing + feature extraction at 4 truncations).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc, math, time, json, collections, urllib.request

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score

torch.manual_seed(2024)
np.random.seed(2024)

device = "cuda" if torch.cuda.is_available() else "cpu"
N_POOL = 200
MAX_LEN = 50
READ_LAYER = 30
TRUNCS = [10, 20, 30, None]     # None = full generated sequence
WARMUP = 3                      # discarded iterations before timing

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("(Record this — absolute timings are hardware-specific; ratios are what transfer.)")

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

class Timer:
    # Context manager that synchronizes CUDA before AND after, so the measured interval
    # covers actual kernel execution rather than async queue submission.
    def __enter__(self):
        sync(); self.t0 = time.perf_counter(); return self
    def __exit__(self, *a):
        sync(); self.dt = time.perf_counter() - self.t0

print(f"Setup ready. N={N_POOL}, max_len={MAX_LEN}, layer={READ_LAYER}, truncations={TRUNCS}")


CUDA available: True
GPU: Tesla T4
(Record this — absolute timings are hardware-specific; ratios are what transfer.)
Setup ready. N=200, max_len=50, layer=30, truncations=[10, 20, 30, None]


In [2]:
# --- Prefix pool, same construction as every ProtGPT2 notebook in this project. ---
UNIPROT = ["P0CG48","P00720","P02144","P42212","P01308","P61823",
           "P00648","P99999","P69905","P68871","P00698","P00441"]
FALLBACK = ["NLYIQWLKDGGPSSGRPPPS","LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF","GSQIGAKNTGQVQLNLLAL",
            "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
            "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
            "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV"]

print("Fetching UniProt reference sequences...")
refs = []
for acc in UNIPROT:
    try:
        with urllib.request.urlopen(f"https://rest.uniprot.org/uniprotkb/{acc}.fasta", timeout=10) as rsp:
            lines = [l for l in rsp.read().decode("utf-8").strip().split("\n") if l]
        s = "".join(lines[1:])
        if len(s) >= 20:
            refs.append((acc, s)); print(f"  {acc}: {len(s)} residues")
    except Exception as e:
        print(f"  skip {acc}: {e}")
if not refs:
    print("!! UniProt fetch failed — Internet toggle likely OFF. Using hardcoded fallback.")
    refs = [(f"local{i+1}", s) for i, s in enumerate(FALLBACK)]

rng = np.random.RandomState(707)
prefixes = []
for i in range(N_POOL):
    acc, s = refs[i % len(refs)]
    pl = rng.randint(10, 16)
    st = rng.randint(0, max(1, len(s) - pl))
    prefixes.append(s[st:st + pl])
print(f"\nBuilt {len(prefixes)} prefixes.")


Fetching UniProt reference sequences...
  P0CG48: 685 residues
  P00720: 164 residues
  P02144: 154 residues
  P42212: 238 residues
  P01308: 110 residues
  P61823: 150 residues
  P00648: 157 residues
  P99999: 105 residues
  P69905: 142 residues
  P68871: 147 residues
  P00698: 147 residues
  P00441: 154 residues

Built 200 prefixes.


In [3]:
# --- TIMING 1: generation. Measured per sequence, with token counts recorded so per-token
#     cost (and therefore the cost of stopping early) is derivable rather than assumed. ---

print(f"Loading ProtGPT2 on {device}...")
tok = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
model.eval()

def generate_one(prompt, max_length):
    inputs = tok(prompt, return_tensors="pt").to(device)
    n_in = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(**inputs, max_length=max_length, do_sample=True,
                             temperature=1.2, pad_token_id=tok.eos_token_id)
    seq = tok.decode(out[0], skip_special_tokens=True).replace(" ", "")
    return seq, n_in, out.shape[1]

print(f"Warm-up ({WARMUP} discarded iterations — first CUDA calls pay one-time autotuning costs)...")
for i in range(WARMUP):
    generate_one(prefixes[i], MAX_LEN)

torch.manual_seed(707)
records, gen_times = [], []
print(f"=== Generating N={N_POOL}, timing each ===")
for i, prompt in enumerate(prefixes):
    with Timer() as t:
        seq, n_in, n_out = generate_one(prompt, MAX_LEN)
    gen_only = seq[len(prompt):] if seq.startswith(prompt) else seq
    records.append({"prompt": prompt, "sequence": seq, "gen_only": gen_only,
                    "n_tok_in": n_in, "n_tok_out": n_out, "gen_time_s": t.dt})
    gen_times.append(t.dt)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{N_POOL}  (running mean {np.mean(gen_times):.3f}s/seq)")

gen_times = np.array(gen_times)
new_tokens = np.array([r["n_tok_out"] - r["n_tok_in"] for r in records])
T_GEN_FULL = float(gen_times.mean())
T_PER_TOKEN = float((gen_times / np.maximum(new_tokens, 1)).mean())

print(f"\nGeneration: {T_GEN_FULL:.4f}s/sequence (sd {gen_times.std():.4f})")
print(f"  new tokens/sequence: mean {new_tokens.mean():.1f}")
print(f"  per new token: {T_PER_TOKEN:.5f}s")
print(f"  generated residues/sequence: mean {np.mean([len(r['gen_only']) for r in records]):.1f}")


Loading ProtGPT2 on cuda...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Warm-up (3 discarded iterations — first CUDA calls pay one-time autotuning costs)...
=== Generating N=200, timing each ===
  50/200  (running mean 0.644s/seq)
  100/200  (running mean 0.583s/seq)
  150/200  (running mean 0.600s/seq)
  200/200  (running mean 0.592s/seq)

Generation: 0.5917s/sequence (sd 0.5095)
  new tokens/sequence: mean 20.5
  per new token: 0.02952s
  generated residues/sequence: mean 49.6


In [4]:
# --- TIMING 2 + labels: ESMFold, per sequence, recorded WITH length (cost scales steeply with
#     length, so a single mean would hide the thing that matters for early-abort accounting). ---

print("=== Freeing ProtGPT2 before loading ESMFold ===")
del model
clear_gpu()

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

print("Loading ESMFold...")
esm_tok = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
esm = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True).to(device).eval()

def fold_one(seq, max_len=300):
    cleaned = "".join(a for a in seq if a in VALID_AA)
    if len(cleaned) < 10:
        return 0.0, 0.0, False, 0
    working = cleaned[:max_len]
    try:
        inputs = esm_tok([working], return_tensors="pt", add_special_tokens=False).to(device)
        with torch.no_grad():
            out = esm(**inputs)
        raw = float(np.mean(out.plddt.cpu().numpy()))
        p = raw * 100.0 if raw <= 1.5 else raw
        t = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
        return p, t, True, len(working)
    except RuntimeError:
        clear_gpu()
        return 0.0, 0.0, False, len(working)

print(f"Warm-up ({WARMUP} discarded)...")
for i in range(WARMUP):
    fold_one(records[i]["sequence"])

fold_times = []
print(f"=== Folding N={N_POOL}, timing each ===")
for i, r in enumerate(records):
    with Timer() as t:
        p, ptm, ok, L = fold_one(r["sequence"])
    r.update({"plddt": p, "ptm": ptm, "fold_ok": ok, "fold_len": L,
              "fold_time_s": t.dt, "collapse": int(0.0 < p < 60.0)})
    if ok:
        fold_times.append((t.dt, L))
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{N_POOL}  (running mean {np.mean([f[0] for f in fold_times]):.3f}s/seq)")

del esm
clear_gpu()

ft = np.array([f[0] for f in fold_times]); fl = np.array([f[1] for f in fold_times])
T_FOLD = float(ft.mean())
valid = [r for r in records if r["fold_ok"]]
collapse_rate = float(np.mean([r["collapse"] for r in valid]))

print(f"\nESMFold: {T_FOLD:.4f}s/sequence (sd {ft.std():.4f}), folded {len(valid)}/{N_POOL}")
print(f"  folded length: mean {fl.mean():.1f} residues (range {fl.min()}-{fl.max()})")
if len(set(fl)) > 3:
    corr = float(np.corrcoef(fl, ft)[0, 1])
    print(f"  corr(length, fold_time) = {corr:+.3f}  <- if strongly positive, fold cost is")
    print(f"     length-driven and the single mean above is a simplification worth stating")
print(f"\nNatural collapse rate: {collapse_rate:.1%}  (§2 reference: 53.2% at N=400)")
print(f"\n*** COST RATIO — the number this notebook exists to produce ***")
print(f"    ESMFold / generation = {T_FOLD / T_GEN_FULL:.2f}x")


=== Freeing ProtGPT2 before loading ESMFold ===
Loading ESMFold...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Warm-up (3 discarded)...
=== Folding N=200, timing each ===
  50/200  (running mean 1.900s/seq)
  100/200  (running mean 2.018s/seq)
  150/200  (running mean 2.060s/seq)
  200/200  (running mean 2.095s/seq)

ESMFold: 2.0948s/sequence (sd 1.5955), folded 198/200
  folded length: mean 58.0 residues (range 10-210)
  corr(length, fold_time) = +0.892  <- if strongly positive, fold cost is
     length-driven and the single mean above is a simplification worth stating

Natural collapse rate: 50.0%  (§2 reference: 53.2% at N=400)

*** COST RATIO — the number this notebook exists to produce ***
    ESMFold / generation = 3.54x


In [5]:
# --- Feature extraction at each truncation, with the classifier's own inference cost timed. ---

print(f"Reloading ProtGPT2 to extract layer-{READ_LAYER} features...")
model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
model.eval()

def feat_one(prompt, gen_only, trunc):
    g = gen_only if trunc is None else gen_only[:trunc]
    full = prompt + g
    if len(full) < 5:
        return None
    inputs = tok(full, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
    return out.hidden_states[READ_LAYER].mean(dim=1).squeeze(0).cpu().numpy()

feats, labels, fwd_times = {}, {}, {}
for tr in TRUNCS:
    for i in range(WARMUP):
        feat_one(valid[i]["prompt"], valid[i]["gen_only"], tr)
    F, Y, TT = [], [], []
    for r in valid:
        with Timer() as t:
            f = feat_one(r["prompt"], r["gen_only"], tr)
        if f is not None:
            F.append(f); Y.append(r["collapse"]); TT.append(t.dt)
    feats[tr], labels[tr], fwd_times[tr] = np.array(F), np.array(Y), float(np.mean(TT))
    lbl = "full" if tr is None else f"{tr}res"
    print(f"  {lbl:>6s}: {len(F)} vectors, forward pass {fwd_times[tr]*1000:.2f} ms/seq")

del model
clear_gpu()
print(f"\nClassifier forward-pass cost is ~{fwd_times[30]*1000:.1f} ms at 30 residues "
      f"vs {T_FOLD*1000:.0f} ms for a fold — a {T_FOLD/fwd_times[30]:.0f}x difference.")


Reloading ProtGPT2 to extract layer-30 features...


Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   10res: 198 vectors, forward pass 26.73 ms/seq
   20res: 198 vectors, forward pass 26.20 ms/seq
   30res: 198 vectors, forward pass 27.11 ms/seq
    full: 198 vectors, forward pass 28.83 ms/seq

Classifier forward-pass cost is ~27.1 ms at 30 residues vs 2095 ms for a fold — a 77x difference.


In [6]:
# --- Out-of-fold predictions + screening curves. Pipeline keeps scaler/PCA inside each CV fold,
#     so no fold's preprocessing sees its own test data (the leak the N=50 preliminary avoided by
#     using a separate training pool; this is the cleaner within-pool version). ---

def oof_probs(X, y, seed=99):
    n_comp = min(20, X.shape[0] - 1, X.shape[1])
    pipe = Pipeline([("sc", StandardScaler()),
                     ("pca", PCA(n_components=n_comp, random_state=seed)),
                     ("rf", RandomForestClassifier(n_estimators=300, random_state=seed))])
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
    return cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]

def screening_curve(prob_collapse, y):
    # rank ascending by predicted collapse prob = most promising first
    order = np.argsort(prob_collapse)
    y_sorted = y[order]
    n = len(y); n_good = int((y == 0).sum())
    base = n_good / n
    rows = []
    for frac in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0):
        k = max(1, int(round(frac * n)))
        found = int((y_sorted[:k] == 0).sum())
        rows.append({"budget": frac, "k": k, "found": found,
                     "recall": found / n_good if n_good else float("nan"),
                     "precision": found / k,
                     "enrichment": (found / (base * k)) if base > 0 else float("nan")})
    return rows, base

print("=" * 92)
print("SCREENING EFFICIENCY BY HOW MUCH OF THE SEQUENCE THE CLASSIFIER SEES")
print("=" * 92)

curves, aucs = {}, {}
for tr in TRUNCS:
    X, y = feats[tr], labels[tr]
    p = oof_probs(X, y)
    aucs[tr] = roc_auc_score(y, p)
    curves[tr], base = screening_curve(p, y)
    lbl = "full" if tr is None else f"{tr} res"
    print(f"\n--- {lbl} (out-of-fold AUC {aucs[tr]:.3f}) ---")
    print(f"{'budget':>7s} {'recall':>8s} {'precision':>10s} {'enrich':>7s}")
    for row in curves[tr]:
        if row["budget"] in (0.1, 0.3, 0.5, 0.7, 0.9):
            print(f"{row['budget']:6.0%} {row['recall']:8.1%} {row['precision']:9.1%} "
                  f"{row['enrichment']:6.2f}x")

print(f"\nBase rate (foldable in pool): {base:.1%}")
print(f"AUC by truncation: " + ", ".join(
    f"{'full' if t is None else str(t)+'res'}={aucs[t]:.3f}" for t in TRUNCS))
print("(§3g reference: 0.682 / 0.720 / 0.740 / 0.780 — close values confirm consistency)")


SCREENING EFFICIENCY BY HOW MUCH OF THE SEQUENCE THE CLASSIFIER SEES

--- 10 res (out-of-fold AUC 0.644) ---
 budget   recall  precision  enrich
   10%    16.2%     80.0%   1.60x
   30%    42.4%     71.2%   1.42x
   50%    58.6%     58.6%   1.17x
   70%    77.8%     55.4%   1.11x
   90%    92.9%     51.7%   1.03x

--- 20 res (out-of-fold AUC 0.645) ---
 budget   recall  precision  enrich
   10%    17.2%     85.0%   1.70x
   30%    41.4%     69.5%   1.39x
   50%    62.6%     62.6%   1.25x
   70%    75.8%     54.0%   1.08x
   90%    90.9%     50.6%   1.01x

--- 30 res (out-of-fold AUC 0.715) ---
 budget   recall  precision  enrich
   10%    17.2%     85.0%   1.70x
   30%    46.5%     78.0%   1.56x
   50%    69.7%     69.7%   1.39x
   70%    80.8%     57.6%   1.15x
   90%    93.9%     52.2%   1.04x

--- full (out-of-fold AUC 0.724) ---
 budget   recall  precision  enrich
   10%    17.2%     85.0%   1.70x
   30%    47.5%     79.7%   1.59x
   50%    65.7%     65.7%   1.31x
   70%    81.8%  

In [7]:
# --- END-TO-END: what a triage policy actually saves, in measured seconds. ---
#
#   baseline(per candidate)      = T_gen_full + T_fold
#   triage(K residues, budget b) = T_gen(K) + T_clf(K) + b * [ (T_gen_full - T_gen(K)) + T_fold ]
#
# T_gen(K) is derived from the measured per-token cost and the observed residues-per-token ratio,
# rather than assumed proportional to residue count.

res_per_seq = float(np.mean([len(r["gen_only"]) for r in valid]))
tok_per_seq = float(np.mean([r["n_tok_out"] - r["n_tok_in"] for r in valid]))
res_per_tok = res_per_seq / max(tok_per_seq, 1e-9)

def t_gen_residues(n_res):
    return min(T_GEN_FULL, T_PER_TOKEN * (n_res / max(res_per_tok, 1e-9)))

BASELINE = T_GEN_FULL + T_FOLD
print("=" * 96)
print("END-TO-END COST OF A TRIAGE POLICY (measured seconds/candidate)")
print("=" * 96)
print(f"Baseline (generate fully + fold every candidate): {BASELINE:.3f} s/candidate")
print(f"  generation {T_GEN_FULL:.3f}s ({T_GEN_FULL/BASELINE:.0%})  |  "
      f"folding {T_FOLD:.3f}s ({T_FOLD/BASELINE:.0%})")
print(f"  ~{res_per_tok:.2f} residues per token")
print()
print(f"{'abort@':>7s} {'budget':>7s} {'recall':>8s} {'cost/cand':>11s} {'saved':>8s} {'AUC':>7s}")
print("-" * 56)

rows_out = []
for tr in TRUNCS:
    if tr is None:
        continue
    tgen_k = t_gen_residues(tr)
    tclf = fwd_times[tr]
    for row in curves[tr]:
        b = row["budget"]
        if b not in (0.3, 0.5, 0.7):
            continue
        cost = tgen_k + tclf + b * ((T_GEN_FULL - tgen_k) + T_FOLD)
        saved = 1.0 - cost / BASELINE
        rows_out.append({"abort_at": tr, "budget": b, "recall": row["recall"],
                         "precision": row["precision"], "cost_per_candidate_s": cost,
                         "frac_saved": saved, "auc": aucs[tr]})
        print(f"{tr:6d}r {b:7.0%} {row['recall']:8.1%} {cost:10.3f}s {saved:7.1%} {aucs[tr]:7.3f}")

print()
print("Reading this table: 'abort@30r, budget 70%' = generate 30 residues, score, keep the most")
print("promising 70%, finish and fold only those. 'recall' = fraction of genuinely foldable")
print("candidates still retained. 'saved' = fraction of total pipeline compute avoided.")

best = max(rows_out, key=lambda r: r["frac_saved"] if r["recall"] >= 0.90 else -1)
if best["recall"] >= 0.90:
    print(f"\n*** Best policy retaining >=90% of foldable candidates: abort at {best['abort_at']} "
          f"residues, fold top {best['budget']:.0%} -> {best['frac_saved']:.0%} of compute saved "
          f"at {best['recall']:.0%} recall. ***")
else:
    print("\nNo tested policy retains >=90% recall; report the recall/savings trade-off curve")
    print("rather than a single headline operating point.")


END-TO-END COST OF A TRIAGE POLICY (measured seconds/candidate)
Baseline (generate fully + fold every candidate): 2.687 s/candidate
  generation 0.592s (22%)  |  folding 2.095s (78%)
  ~2.34 residues per token

 abort@  budget   recall   cost/cand    saved     AUC
--------------------------------------------------------
    10r     30%    42.4%      0.921s   65.7%   0.644
    10r     50%    58.6%      1.433s   46.7%   0.644
    10r     70%    77.8%      1.945s   27.6%   0.644
    20r     30%    41.4%      1.009s   62.4%   0.645
    20r     50%    62.6%      1.496s   44.3%   0.645
    20r     70%    75.8%      1.983s   26.2%   0.645
    30r     30%    46.5%      1.098s   59.1%   0.715
    30r     50%    69.7%      1.560s   41.9%   0.715
    30r     70%    80.8%      2.021s   24.8%   0.715

Reading this table: 'abort@30r, budget 70%' = generate 30 residues, score, keep the most
promising 70%, finish and fold only those. 'recall' = fraction of genuinely foldable
candidates still retained.

In [8]:
# --- Persist everything. ---
pd.DataFrame([{
    "gpu": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
    "n_pool": N_POOL, "n_folded": len(valid), "collapse_rate": collapse_rate,
    "t_gen_full_s": T_GEN_FULL, "t_per_token_s": T_PER_TOKEN, "t_fold_s": T_FOLD,
    "fold_over_gen_ratio": T_FOLD / T_GEN_FULL,
    "baseline_s_per_candidate": BASELINE,
    "residues_per_token": res_per_tok,
    **{f"t_clf_{('full' if t is None else t)}_s": fwd_times[t] for t in TRUNCS},
    **{f"auc_{('full' if t is None else t)}": aucs[t] for t in TRUNCS},
}]).to_csv("compute_cost_summary.csv", index=False)

pd.DataFrame(rows_out).to_csv("compute_triage_policies.csv", index=False)

curve_rows = []
for tr in TRUNCS:
    for row in curves[tr]:
        curve_rows.append({"truncation": (-1 if tr is None else tr), **row})
pd.DataFrame(curve_rows).to_csv("screening_curves_by_truncation.csv", index=False)

pd.DataFrame([{
    "prompt": r["prompt"], "sequence": r["sequence"], "gen_only": r["gen_only"],
    "gen_time_s": r["gen_time_s"], "fold_time_s": r["fold_time_s"], "fold_len": r["fold_len"],
    "n_tok_out": r["n_tok_out"], "plddt": r["plddt"], "ptm": r["ptm"],
    "fold_ok": r["fold_ok"], "collapse": r["collapse"],
} for r in records]).to_csv("compute_timing_per_sequence.csv", index=False)

print("Saved:")
print("  compute_cost_summary.csv          <- the headline ratios + GPU name")
print("  compute_triage_policies.csv       <- cost/recall/savings per policy")
print("  screening_curves_by_truncation.csv")
print("  compute_timing_per_sequence.csv   <- raw per-sequence timings")
print()
print("=" * 78)
print("WHAT TO DO WITH THIS")
print("=" * 78)
print("1. Add a new section to notes/locked-results.md (suggest §3j) with the cost ratio and the")
print("   best triage policy. Report the GPU name alongside — absolute seconds are")
print("   hardware-specific, the ratio is what transfers.")
print("2. This is the Track 4 'decision-aware metrics' and Track 5 'resource-constrained")
print("   discovery' material — currently the paper has no Track 5 content at all.")
print("3. Cross-check the AUC row against §3g (0.682/0.720/0.740/0.780). Large divergence would")
print("   mean this pool differs from §3g's; small divergence confirms both.")


Saved:
  compute_cost_summary.csv          <- the headline ratios + GPU name
  compute_triage_policies.csv       <- cost/recall/savings per policy
  screening_curves_by_truncation.csv
  compute_timing_per_sequence.csv   <- raw per-sequence timings

WHAT TO DO WITH THIS
1. Add a new section to notes/locked-results.md (suggest §3j) with the cost ratio and the
   best triage policy. Report the GPU name alongside — absolute seconds are
   hardware-specific, the ratio is what transfers.
2. This is the Track 4 'decision-aware metrics' and Track 5 'resource-constrained
   discovery' material — currently the paper has no Track 5 content at all.
3. Cross-check the AUC row against §3g (0.682/0.720/0.740/0.780). Large divergence would
   mean this pool differs from §3g's; small divergence confirms both.
